# Bài thực hành 2 — Phân loại tuyến tính

MSSV: 24521185  
Họ và tên: Nguyễn Anh Nguyên

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import os
import cv2
import numpy as np
from PIL import Image

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score
from sklearn.multiclass import OneVsRestClassifier

## Đọc dữ liệu và đặc trưng X từ bài thực hành 1

X là histogram HSV 8×8×8, đạt Accuracy 62/92 = 0,6739 trong bài 1. Giữ nguyên cách trích đặc trưng và tập train/test.

In [2]:
def load_images_from_folder(folder):
    images, labels, paths = [], [], []
    class_names = sorted(name for name in os.listdir(folder) if os.path.isdir(os.path.join(folder, name)))
    for label, class_name in enumerate(class_names):
        class_folder = os.path.join(folder, class_name)
        for filename in sorted(os.listdir(class_folder)):
            if not filename.lower().endswith('.jpg'):
                continue
            image_path = os.path.join(class_folder, filename)
            image = cv2.imread(image_path)
            if image is None:
                image = cv2.cvtColor(np.array(Image.open(image_path).convert('RGB')), cv2.COLOR_RGB2BGR)
            images.append(image)
            labels.append(label)
            paths.append(image_path)
    return images, np.array(labels), paths, class_names

dataset_folder = '/content/drive/MyDrive/CS231/Dataset/HoaVietNam2027'
train_folder = os.path.join(dataset_folder, 'train')
test_folder = os.path.join(dataset_folder, 'test')
train_images, y_train, _, class_names = load_images_from_folder(train_folder)
test_images, y_test, test_paths, test_class_names = load_images_from_folder(test_folder)
assert class_names == test_class_names
print('Các lớp:', class_names)
print('Train:', len(train_images), '| Test:', len(test_images))

Các lớp: ['Cuc', 'Dao', 'Ga', 'Hong', 'Lan', 'Mai', 'Sen', 'Tho']
Train: 285 | Test: 92


In [3]:
X_NAME = 'Histogram HSV 8×8×8'

def compute_histogram_hsv(image, bins=8):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([image], [0, 1, 2], None, [bins, bins, bins],
                        [0, 180, 0, 256, 0, 256])
    return cv2.normalize(hist, hist).flatten()

X_train = np.array([compute_histogram_hsv(image) for image in train_images])
X_test = np.array([compute_histogram_hsv(image) for image in test_images])
print('Đặc trưng X:', X_NAME)
print('Kích thước:', X_train.shape, X_test.shape)

Đặc trưng X: Histogram HSV 8×8×8
Kích thước: (285, 512) (92, 512)


## Yêu cầu 1 — LogisticRegression: chọn penalty và solver theo macro-F1

Tìm tham số bằng kiểm định chéo 5 fold trên train. Với liblinear, dùng One-vs-Rest để phân loại tám lớp; các solver còn lại dùng LogisticRegression đa lớp. Elasticnet dùng l1_ratio = 0,5.


In [4]:
param_grid = [
    {'penalty': ['l1'], 'solver': ['saga']},
    {'penalty': ['l2'], 'solver': ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']},
    {'penalty': ['elasticnet'], 'solver': ['saga'], 'l1_ratio': [0.5]},
    {'penalty': [None], 'solver': ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']},
]
search_lr = GridSearchCV(LogisticRegression(max_iter=5000, random_state=42),
                         param_grid, scoring='f1_macro', cv=5, n_jobs=1, error_score='raise')
search_lr.fit(X_train, y_train)
search_liblinear = GridSearchCV(
    OneVsRestClassifier(LogisticRegression(solver='liblinear', max_iter=5000, random_state=42)),
    {'estimator__penalty': ['l1', 'l2']}, scoring='f1_macro', cv=5, n_jobs=1, error_score='raise')
search_liblinear.fit(X_train, y_train)

if search_liblinear.best_score_ > search_lr.best_score_:
    best_model_lr = search_liblinear.best_estimator_
    best_params_lr = {'solver': 'liblinear', 'penalty': search_liblinear.best_params_['estimator__penalty']}
    best_cv_lr = search_liblinear.best_score_
else:
    best_model_lr = search_lr.best_estimator_
    best_params_lr = search_lr.best_params_
    best_cv_lr = search_lr.best_score_

y_pred_lr = best_model_lr.predict(X_test)
print('Tham số tốt nhất:', best_params_lr)
print(f'Macro-F1 CV: {best_cv_lr:.4f}')
print(f'Macro-F1 test: {f1_score(y_test, y_pred_lr, average="macro"):.4f}')
print(classification_report(y_test, y_pred_lr, target_names=class_names, zero_division=0))


Tham số tốt nhất: {'penalty': 'l2', 'solver': 'lbfgs'}
Macro-F1 CV: 0.6372
Macro-F1 test: 0.5951
              precision    recall  f1-score   support

         Cuc       0.50      0.70      0.58        10
         Dao       0.59      0.83      0.69        12
          Ga       0.56      0.67      0.61        15
        Hong       0.33      0.13      0.19        15
         Lan       1.00      0.40      0.57        10
         Mai       0.80      0.80      0.80        10
         Sen       0.50      0.70      0.58        10
         Tho       0.78      0.70      0.74        10

    accuracy                           0.60        92
   macro avg       0.63      0.62      0.60        92
weighted avg       0.61      0.60      0.58        92



## Yêu cầu 2 — SGDClassifier: chọn tham số theo macro-F1

SGDClassifier không có tham số solver. Thử loss, penalty và alpha bằng GridSearchCV với cùng cách chia 5 fold và tiêu chí macro-F1.

In [5]:
param_sgd = {
    'loss': ['hinge', 'log_loss'],
    'penalty': ['l1', 'l2', 'elasticnet', None],
    'alpha': [1e-4, 1e-3, 1e-2],
}
search_sgd = GridSearchCV(SGDClassifier(max_iter=2000, random_state=42),
                          param_sgd, scoring='f1_macro', cv=5, n_jobs=1)
search_sgd.fit(X_train, y_train)
y_pred_sgd = search_sgd.best_estimator_.predict(X_test)
print('Tham số tốt nhất:', search_sgd.best_params_)
print(f'Macro-F1 CV: {search_sgd.best_score_:.4f}')
print(f'Macro-F1 test: {f1_score(y_test, y_pred_sgd, average="macro"):.4f}')
print(classification_report(y_test, y_pred_sgd, target_names=class_names, zero_division=0))

Tham số tốt nhất: {'alpha': 0.01, 'loss': 'hinge', 'penalty': None}
Macro-F1 CV: 0.6659
Macro-F1 test: 0.6528
              precision    recall  f1-score   support

         Cuc       0.55      0.60      0.57        10
         Dao       0.62      0.83      0.71        12
          Ga       0.67      0.80      0.73        15
        Hong       0.50      0.33      0.40        15
         Lan       0.67      0.60      0.63        10
         Mai       0.80      0.80      0.80        10
         Sen       0.60      0.60      0.60        10
         Tho       0.88      0.70      0.78        10

    accuracy                           0.65        92
   macro avg       0.66      0.66      0.65        92
weighted avg       0.65      0.65      0.64        92



In [6]:
print('Nhận xét và kết luận')
print(f'Đặc trưng X: {X_NAME}.')
print(f'LogisticRegression: macro-F1 test = {f1_score(y_test, y_pred_lr, average="macro"):.4f}, tham số tốt nhất = {best_params_lr}.')
print(f'SGDClassifier: macro-F1 test = {f1_score(y_test, y_pred_sgd, average="macro"):.4f}, tham số tốt nhất = {search_sgd.best_params_}.')

Nhận xét và kết luận
Đặc trưng X: Histogram HSV 8×8×8.
LogisticRegression: macro-F1 test = 0.5951, tham số tốt nhất = {'penalty': 'l2', 'solver': 'lbfgs'}.
SGDClassifier: macro-F1 test = 0.6528, tham số tốt nhất = {'alpha': 0.01, 'loss': 'hinge', 'penalty': None}.
